In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/house-price-prediction-challenge-regression/estate_train.csv
/kaggle/input/competitions/house-price-prediction-challenge-regression/estate_sample_submission.csv
/kaggle/input/competitions/house-price-prediction-challenge-regression/estate_test.csv


In [2]:
# # ============================================
# # 2. LOAD DATA
# # ============================================

# DATA_PATH = "/kaggle/input/competitions/house-price-prediction-challenge-regression/"

# # Training data
# train = pd.read_csv(DATA_PATH + "estate_train.csv")

# # Test data
# test = pd.read_csv(DATA_PATH + "estate_test.csv")

# # Sample submission
# sample_submission = pd.read_csv(DATA_PATH + "estate_sample_submission.csv")

# print("Dataset loaded successfully")


In [3]:
# ============================================================
# HOUSE PRICE PREDICTION CHALLENGE
# FIRST SUBMISSION - COMPLETE PIPELINE
# ============================================================

# =========================
# 1. IMPORTS
# =========================

import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor


# =========================
# 2. PATHS
# =========================

BASE_PATH = "/kaggle/input/competitions/house-price-prediction-challenge-regression"

TRAIN_PATH = os.path.join(BASE_PATH, "estate_train.csv")
TEST_PATH = os.path.join(BASE_PATH, "estate_test.csv")
SAMPLE_PATH = os.path.join(BASE_PATH, "estate_sample_submission.csv")

OUTPUT_PATH = "/kaggle/working/submission.csv"


# =========================
# 3. LOAD DATA
# =========================

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
sample = pd.read_csv(SAMPLE_PATH)

print("=" * 60)
print("DATA LOADED")
print("=" * 60)

print("Train shape:", train.shape)
print("Test shape :", test.shape)
print("Sample shape:", sample.shape)

print("\nTrain columns:")
print(train.columns.tolist())


# =========================
# 4. BASIC CHECKS
# =========================

TARGET = "TargetPrice"
ID_COL = "PropertyID"

print("\nMissing values:")
print(train.isnull().sum())

print("\nTarget statistics:")
print(train[TARGET].describe())


# =========================
# 5. SEPARATE FEATURES / TARGET
# =========================

X = train.drop(columns=[TARGET, ID_COL]).copy()
y = train[TARGET].copy()

X_test = test.drop(columns=[ID_COL]).copy()

test_ids = test[ID_COL].copy()


# =========================
# 6. FEATURE ENGINEERING
# =========================
#
# The dataset already contains:
# RoomsPerHousehold
# BedroomsRatio
#
# We preserve these and add a few robust interaction features.
# =========================

def create_features(df):
    df = df.copy()

    # -------------------------
    # Basic ratios
    # -------------------------

    # Income per occupancy
    df["IncomePerOccupant"] = (
        df["IncomeLevel"] /
        (df["AvgOccupancy"].abs() + 1e-3)
    )

    # Population per occupancy
    df["PopulationPerOccupancy"] = (
        df["NeighborhoodPop"] /
        (df["AvgOccupancy"].abs() + 1e-3)
    )

    # Rooms relative to bedrooms
    df["RoomsToBedrooms"] = (
        df["TotalRooms"] /
        (df["TotalBedrooms"].abs() + 1e-3)
    )

    # Population relative to rooms
    df["PopulationPerRoom"] = (
        df["NeighborhoodPop"] /
        (df["TotalRooms"].abs() + 1e-3)
    )

    # -------------------------
    # Geographic interaction
    # -------------------------

    df["LatLon"] = df["Latitude"] * df["Longitude"]

    df["Latitude2"] = df["Latitude"] ** 2
    df["Longitude2"] = df["Longitude"] ** 2

    # Distance-like geographic feature
    df["GeoDistance"] = np.sqrt(
        df["Latitude"] ** 2 +
        df["Longitude"] ** 2
    )

    # -------------------------
    # Non-linear income features
    # -------------------------

    df["IncomeSquared"] = df["IncomeLevel"] ** 2
    df["IncomeLog"] = np.log1p(
        np.clip(df["IncomeLevel"], 0, None)
    )

    # -------------------------
    # Room features
    # -------------------------

    df["RoomsSquared"] = df["TotalRooms"] ** 2
    df["BedroomsSquared"] = df["TotalBedrooms"] ** 2

    # -------------------------
    # Occupancy features
    # -------------------------

    df["OccupancySquared"] = df["AvgOccupancy"] ** 2

    # -------------------------
    # Population transformation
    # -------------------------

    df["PopulationLog"] = np.log1p(
        np.clip(df["NeighborhoodPop"], 0, None)
    )

    return df


X = create_features(X)
X_test = create_features(X_test)


# =========================
# 7. HANDLE MISSING VALUES
# =========================

# PropertyAge is the known missing feature.
# Median imputation is robust to outliers.

property_age_median = X["PropertyAge"].median()

X["PropertyAge"] = X["PropertyAge"].fillna(property_age_median)
X_test["PropertyAge"] = X_test["PropertyAge"].fillna(property_age_median)

# Safety: handle any other accidental missing values
for col in X.columns:
    if X[col].isnull().any():
        median_value = X[col].median()

        X[col] = X[col].fillna(median_value)
        X_test[col] = X_test[col].fillna(median_value)


# =========================
# 8. CLIP EXTREME NUMERIC VALUES
# =========================
#
# Avoid pathological effects from injected/outlier values.
# We calculate limits from training data only.
# =========================

for col in X.columns:

    lower = X[col].quantile(0.001)
    upper = X[col].quantile(0.999)

    X[col] = X[col].clip(lower, upper)
    X_test[col] = X_test[col].clip(lower, upper)


print("\nFinal feature count:", X.shape[1])
print("Final training shape:", X.shape)
print("Final test shape:", X_test.shape)


# =========================
# 9. CROSS VALIDATION
# =========================

N_SPLITS = 5

kf = KFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=42
)


# =========================
# 10. MODEL CONFIGURATION
# =========================

models_config = {

    "CatBoost": {
        "iterations": 1800,
        "depth": 7,
        "learning_rate": 0.03,
        "loss_function": "RMSE",
        "l2_leaf_reg": 5,
        "random_seed": 42,
        "verbose": False,
        "allow_writing_files": False
    },

    "XGBoost": {
        "n_estimators": 1800,
        "max_depth": 7,
        "learning_rate": 0.03,
        "subsample": 0.90,
        "colsample_bytree": 0.95,
        "min_child_weight": 3,
        "reg_lambda": 1,
        "objective": "reg:squarederror",
        "eval_metric": "rmse",
        "random_state": 42,
        "n_jobs": -1
    },

    "LightGBM": {
        "n_estimators": 1800,
        "learning_rate": 0.03,
        "num_leaves": 31,
        "min_child_samples": 20,
        "subsample": 0.90,
        "colsample_bytree": 0.95,
        "reg_lambda": 0.2,
        "verbosity": -1,
        "random_state": 42,
        "n_jobs": -1
    }
}


# =========================
# 11. ENSEMBLE WEIGHTS
# =========================
#
# Based on our validation experiments:
#
# CatBoost  : 40%
# XGBoost   : 40%
# LightGBM  : 20%
#
# This blend performed better than individual models.
# =========================

WEIGHTS = {
    "CatBoost": 0.40,
    "XGBoost": 0.40,
    "LightGBM": 0.20
}


# =========================
# 12. STORAGE
# =========================

oof_predictions = {
    "CatBoost": np.zeros(len(X)),
    "XGBoost": np.zeros(len(X)),
    "LightGBM": np.zeros(len(X))
}

test_predictions = {
    "CatBoost": np.zeros(len(X_test)),
    "XGBoost": np.zeros(len(X_test)),
    "LightGBM": np.zeros(len(X_test))
}

fold_scores = {
    "CatBoost": [],
    "XGBoost": [],
    "LightGBM": []
}


# =========================
# 13. 5-FOLD TRAINING
# =========================

print("\n")
print("=" * 60)
print("5-FOLD ENSEMBLE TRAINING")
print("=" * 60)


for fold, (train_idx, valid_idx) in enumerate(
    kf.split(X),
    start=1
):

    print(f"\n{'=' * 25}")
    print(f"FOLD {fold}/{N_SPLITS}")
    print(f"{'=' * 25}")

    X_train = X.iloc[train_idx]
    X_valid = X.iloc[valid_idx]

    y_train = y.iloc[train_idx]
    y_valid = y.iloc[valid_idx]


    # --------------------------------------------------------
    # CATBOOST
    # --------------------------------------------------------

    print("\nTraining CatBoost...")

    cat_model = CatBoostRegressor(
        **models_config["CatBoost"]
    )

    cat_model.fit(
        X_train,
        y_train,
        eval_set=(X_valid, y_valid),
        early_stopping_rounds=150,
        verbose=False
    )

    cat_valid_pred = cat_model.predict(X_valid)
    cat_test_pred = cat_model.predict(X_test)

    cat_rmse = np.sqrt(
        mean_squared_error(y_valid, cat_valid_pred)
    )

    oof_predictions["CatBoost"][valid_idx] = cat_valid_pred
    test_predictions["CatBoost"] += (
        cat_test_pred / N_SPLITS
    )

    fold_scores["CatBoost"].append(cat_rmse)

    print(f"CatBoost RMSE: {cat_rmse:.6f}")


    # --------------------------------------------------------
    # XGBOOST
    # --------------------------------------------------------

    print("\nTraining XGBoost...")

    xgb_model = XGBRegressor(
        **models_config["XGBoost"]
    )

    xgb_model.fit(
        X_train,
        y_train,
        eval_set=[(X_valid, y_valid)],
        verbose=False
    )

    xgb_valid_pred = xgb_model.predict(X_valid)
    xgb_test_pred = xgb_model.predict(X_test)

    xgb_rmse = np.sqrt(
        mean_squared_error(y_valid, xgb_valid_pred)
    )

    oof_predictions["XGBoost"][valid_idx] = xgb_valid_pred
    test_predictions["XGBoost"] += (
        xgb_test_pred / N_SPLITS
    )

    fold_scores["XGBoost"].append(xgb_rmse)

    print(f"XGBoost RMSE: {xgb_rmse:.6f}")


    # --------------------------------------------------------
    # LIGHTGBM
    # --------------------------------------------------------

    print("\nTraining LightGBM...")

    lgb_model = LGBMRegressor(
        **models_config["LightGBM"]
    )

    lgb_model.fit(
        X_train,
        y_train,
        eval_set=[(X_valid, y_valid)],
        callbacks=[]
    )

    lgb_valid_pred = lgb_model.predict(X_valid)
    lgb_test_pred = lgb_model.predict(X_test)

    lgb_rmse = np.sqrt(
        mean_squared_error(y_valid, lgb_valid_pred)
    )

    oof_predictions["LightGBM"][valid_idx] = lgb_valid_pred
    test_predictions["LightGBM"] += (
        lgb_test_pred / N_SPLITS
    )

    fold_scores["LightGBM"].append(lgb_rmse)

    print(f"LightGBM RMSE: {lgb_rmse:.6f}")


# =========================
# 14. INDIVIDUAL OOF SCORES
# =========================

print("\n")
print("=" * 60)
print("CROSS-VALIDATION RESULTS")
print("=" * 60)

for model_name in oof_predictions:

    score = np.sqrt(
        mean_squared_error(
            y,
            oof_predictions[model_name]
        )
    )

    print(
        f"{model_name:12s} "
        f"OOF RMSE = {score:.6f}"
    )


# =========================
# 15. ENSEMBLE OOF PREDICTION
# =========================

ensemble_oof = (
    WEIGHTS["CatBoost"] *
    oof_predictions["CatBoost"]

    +

    WEIGHTS["XGBoost"] *
    oof_predictions["XGBoost"]

    +

    WEIGHTS["LightGBM"] *
    oof_predictions["LightGBM"]
)


ensemble_rmse = np.sqrt(
    mean_squared_error(
        y,
        ensemble_oof
    )
)


print("\n" + "=" * 60)
print("ENSEMBLE RESULT")
print("=" * 60)

print(
    f"Ensemble OOF RMSE: {ensemble_rmse:.6f}"
)


# =========================
# 16. CREATE TEST PREDICTION
# =========================

final_predictions = (
    WEIGHTS["CatBoost"] *
    test_predictions["CatBoost"]

    +

    WEIGHTS["XGBoost"] *
    test_predictions["XGBoost"]

    +

    WEIGHTS["LightGBM"] *
    test_predictions["LightGBM"]
)


# =========================
# 17. TARGET BOUNDARY
# =========================
#
# Training target has an upper boundary around 5.00001.
# Do not allow impossible negative predictions.
# =========================

final_predictions = np.clip(
    final_predictions,
    0,
    5.00001
)


# =========================
# 18. CREATE SUBMISSION
# =========================

submission = pd.DataFrame({
    "PropertyID": test_ids,
    "TargetPrice": final_predictions
})


# Make absolutely sure column order is correct
submission = submission[
    ["PropertyID", "TargetPrice"]
]


# =========================
# 19. VALIDATE SUBMISSION
# =========================

print("\n")
print("=" * 60)
print("SUBMISSION CHECK")
print("=" * 60)

print("Submission shape:", submission.shape)

print("\nSubmission columns:")
print(submission.columns.tolist())

print("\nFirst 10 predictions:")
print(submission.head(10))

print("\nPrediction statistics:")
print(submission["TargetPrice"].describe())

print("\nMissing values:")
print(submission.isnull().sum())

print(
    "\nUnique PropertyIDs:",
    submission["PropertyID"].nunique()
)

print(
    "Expected PropertyIDs:",
    test[ID_COL].nunique()
)


# =========================
# 20. SAVE SUBMISSION
# =========================

submission.to_csv(
    OUTPUT_PATH,
    index=False
)


# =========================
# 21. FINAL VERIFICATION
# =========================

check = pd.read_csv(OUTPUT_PATH)

assert list(check.columns) == [
    "PropertyID",
    "TargetPrice"
]

assert len(check) == len(test)

assert check["PropertyID"].equals(
    test["PropertyID"]
)

assert check["TargetPrice"].notna().all()

print("\n")
print("=" * 60)
print("SUBMISSION READY")
print("=" * 60)

print("File:", OUTPUT_PATH)
print("Rows:", len(check))
print("Columns:", list(check.columns))

print("\nFirst 5 rows:")
print(check.head())

print("\nDONE.")
print("Download/use this file:")
print(OUTPUT_PATH)

DATA LOADED
Train shape: (16512, 12)
Test shape : (4128, 11)
Sample shape: (4128, 2)

Train columns:
['IncomeLevel', 'PropertyAge', 'TotalRooms', 'TotalBedrooms', 'NeighborhoodPop', 'AvgOccupancy', 'Latitude', 'Longitude', 'TargetPrice', 'PropertyID', 'RoomsPerHousehold', 'BedroomsRatio']

Missing values:
IncomeLevel             0
PropertyAge          1313
TotalRooms              0
TotalBedrooms           0
NeighborhoodPop         0
AvgOccupancy            0
Latitude                0
Longitude               0
TargetPrice             0
PropertyID              0
RoomsPerHousehold       0
BedroomsRatio           0
dtype: int64

Target statistics:
count    16512.000000
mean         2.071947
std          1.156226
min          0.149990
25%          1.198000
50%          1.798500
75%          2.651250
max          5.000010
Name: TargetPrice, dtype: float64

Final feature count: 24
Final training shape: (16512, 24)
Final test shape: (4128, 24)


5-FOLD ENSEMBLE TRAINING

FOLD 1/5

Training Cat